# Audio Feature Extraction: From Waveform to Spectrogram

---

## Why Do We Need to Transform Audio Before Feeding It Into a Neural Network?

Imagine recording 1 second of speech at a sample rate of 16,000 Hz.  
You get **16,000 numbers** — each one is the air pressure amplitude at a tiny moment in time.

**Problems with raw waveform:**
- Individual sample values carry little meaning — hearing perceives sound through *frequency*, not raw amplitude
- Enormous number of dimensions makes training difficult
- Not invariant to phase shift — the same sound starting at a different time looks completely different

**Solution:** Convert to a **Spectrogram** — a 2D representation showing which frequencies are present at each moment in time, and how loud they are.  
This is essentially what the human auditory system does naturally.

```
Raw Waveform  (time domain)
    │  STFT
    ▼
Spectrogram  (time x frequency)
    │  Mel filterbank
    ▼
Mel Spectrogram  (time x mel frequency)
    │  log()
    ▼
Log-Mel Spectrogram  <- what most deep learning models use as input
    │  DCT
    ▼
MFCC  <- used in traditional speech recognition
```

In [ ]:
# !pip install librosa soundfile matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from IPython.display import Audio
import warnings
warnings.filterwarnings('ignore')

print(f"librosa version: {librosa.__version__}")

## 1. What Is Sound? (Time Domain)

Sound is **vibration of air** — changes in air pressure over time.  
When a microphone records audio, it measures amplitude (pressure) at discrete points in time.  
The number of measurements per second is the **Sample Rate** (Hz).

![Signal Sampling](./figures/Signal_Sampling.png)

**Common sample rates:**

| Sample Rate | Used For |
|-------------|---------|
| 8,000 Hz | Telephone (covers speech range) |
| 16,000 Hz | Speech recognition — Whisper, DeepSpeech |
| 22,050 Hz | TTS — Tacotron 2, WaveNet |
| 44,100 Hz | CD-quality music |
| 48,000 Hz | Video production |

**Nyquist Theorem:** To capture frequencies up to `f_max` Hz correctly,  
you need a sample rate of at least `2 x f_max`.  
Humans hear up to ~20 kHz, so music needs sr >= 40 kHz.

In [ ]:
# Load audio (uses h_1.wav if present, otherwise generates a synthetic signal)
try:
    y, sr = librosa.load('h_1.wav', sr=16000)
    print(f"Loaded h_1.wav: {len(y)} samples @ {sr} Hz = {len(y)/sr:.2f} seconds")
except Exception:
    # Synthetic: three harmonics mixed together, like a vowel sound
    sr = 16000
    t  = np.linspace(0, 1.0, sr)
    y  = (0.5 * np.sin(2 * np.pi * 440  * t) +   # fundamental (A4)
          0.3 * np.sin(2 * np.pi * 880  * t) +   # 2nd harmonic
          0.2 * np.sin(2 * np.pi * 1320 * t))    # 3rd harmonic
    y  = y.astype(np.float32)
    print(f"Using synthetic signal: {len(y)} samples @ {sr} Hz")

duration = len(y) / sr

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.linspace(0, duration, len(y)), y, color='steelblue', linewidth=0.5, alpha=0.8)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Waveform — {duration:.2f}s, sample rate = {sr:,} Hz')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Array shape:      {y.shape}")
print(f"Value range:      [{y.min():.3f}, {y.max():.3f}]")
print(f"First 10 samples: {y[:10].round(4)}")

## 2. Fourier Transform — Decomposing a Signal into Frequencies

**Question:** Given a raw waveform, how do we find out which frequencies are present?

**Answer:** The Fourier Transform — converts a signal from the **time domain** to the **frequency domain**.

The key insight: any signal can be expressed as a sum of sine waves at different frequencies.  
The Fourier Transform finds the amplitude and phase of each frequency component.

```
f(t) = sum_k  A_k * sin(2*pi * f_k * t + phi_k)
               ^           ^               ^
           amplitude    frequency        phase
```

**Limitation of the plain DFT:** it operates on the whole signal at once.  
It tells you *which* frequencies exist — but not *when* they occur.  
Speech has frequencies that change over time (consonants vs. vowels) — we need time AND frequency together.

**Solution: Short-Time Fourier Transform (STFT)**

## 3. STFT — Short-Time Fourier Transform

**Idea:** Instead of running FFT on the entire signal, divide it into short overlapping **frames** (e.g. 25 ms) and apply FFT independently to each frame.

![STFT Concept](./figures/stft_concept.png)

**Key parameters:**

| Parameter | librosa arg | Meaning | Typical value |
|-----------|-------------|---------|--------------|
| Window size | `n_fft` | Samples per frame | 512–2048 |
| Hop length | `hop_length` | Step between windows | n_fft / 4 |
| Window function | `window` | Smooths frame edges | `hann` |

**Why the Hann window?**  
Cutting a signal at frame boundaries with a rectangular window introduces **spectral leakage** — fake frequency components appear in the FFT result.  
The Hann window tapers smoothly to zero at both edges, suppressing this artifact.

```
Frame j (25 ms):
  [ 0, x_1, x_2, ..., x_N, 0 ]
     ^                    ^
  hann window tapers to zero at edges

STFT(j, k) = sum_n  x[n] * w[n] * exp(-i * 2*pi * k * n / N)
```

**Output shape:**  
`STFT.shape = (n_fft/2 + 1, n_frames)`  
- Rows = frequency bins from 0 Hz to sr/2 Hz  
- Cols = time frames

In [ ]:
# ── Short-Time Fourier Transform ──

n_fft      = 1024   # window: 1024 samples = 64 ms at 16 kHz
hop_length = 256    # hop:    256 samples  = 16 ms  (75% overlap)
window     = 'hann'

D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, window=window)
# D is complex — magnitude encodes energy, phase encodes timing

S = np.abs(D) ** 2   # power spectrogram

print(f"STFT shape:           {D.shape}   (freq_bins x time_frames)")
print(f"Frequency bins:       {D.shape[0]}  (= n_fft/2 + 1)")
print(f"Time frames:          {D.shape[1]}")
print(f"Frequency resolution: {sr/n_fft:.1f} Hz per bin")
print(f"Time resolution:      {hop_length/sr*1000:.1f} ms per frame")
print(f"Frequency range:      0 to {sr//2:,} Hz")

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(np.linspace(0, duration, len(y)), y, color='steelblue', linewidth=0.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Waveform (Time Domain)')
axes[0].set_xlim(0, duration)

img = librosa.display.specshow(S, sr=sr, hop_length=hop_length,
                               x_axis='time', y_axis='linear', ax=axes[1])
axes[1].set_title('Power Spectrogram — Linear Frequency Scale')
fig.colorbar(img, ax=axes[1], format='%+2.0f')

plt.tight_layout(); plt.show()

print("\nNote: most energy sits at low frequencies.")
print("The linear scale wastes space on high frequencies we barely perceive.")

## 4. Why the Mel Scale?

**Problem:** The human ear does not hear frequency linearly.

We easily distinguish 100 Hz from 200 Hz.  
But we can barely tell apart 10,000 Hz from 10,100 Hz —  
even though both pairs differ by exactly 100 Hz.

**The Mel Scale** models human auditory perception:  
two sounds separated by 1 mel are perceived as equally distant anywhere on the scale.

$$m = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

| Linear Hz | Mel |
|-----------|-----|
| 100 Hz | ~150 mel |
| 1,000 Hz | ~1,000 mel |
| 4,000 Hz | ~2,146 mel |
| 10,000 Hz | ~2,840 mel |

**Mel Filterbank:** a bank of triangular filters spaced logarithmically across the frequency range.  
Each filter sums the energy inside its band — so higher frequencies get wider (coarser) filters.

![Mel Filterbank](./figures/mel_filterbank.png)

In [ ]:
# ── Mel Filterbank ──

n_mels = 80   # number of mel bands (40–128 are common for speech)

mel_filter = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels,
                                  fmin=0.0, fmax=sr // 2)

print(f"Mel filterbank shape: {mel_filter.shape}")
print(f"  {n_mels} filters  x  {mel_filter.shape[1]} frequency bins")

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

freq_bins = np.linspace(0, sr / 2, n_fft // 2 + 1)
for i in range(0, n_mels, 5):
    axes[0].plot(freq_bins, mel_filter[i], alpha=0.6)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Filter Weight')
axes[0].set_title(f'Mel Filterbank — {n_mels} triangular filters (every 5th shown)')
axes[0].set_xlim(0, sr // 2)

img = librosa.display.specshow(mel_filter, sr=sr, x_axis='linear', ax=axes[1])
fig.colorbar(img, ax=axes[1])
axes[1].set_title('Mel Filterbank (image view) — filters widen at high frequencies')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Mel Filter Index')

plt.tight_layout(); plt.show()

# Manually applying the filterbank = matrix multiply
mel_S = mel_filter @ S
print(f"\nMel Spectrogram shape: {mel_S.shape}  (n_mels x n_frames)")

In [ ]:
# ── Mel Spectrogram ──
# librosa.feature.melspectrogram runs STFT + filterbank in one step

mel_S = librosa.feature.melspectrogram(
    y=y, sr=sr,
    n_fft=n_fft, hop_length=hop_length,
    n_mels=n_mels, fmin=0.0, fmax=sr // 2,
    power=2.0   # energy = |STFT|^2
)

print(f"Mel Spectrogram shape: {mel_S.shape}  ({n_mels} bands x {mel_S.shape[1]} frames)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

librosa.display.specshow(librosa.amplitude_to_db(np.abs(D), ref=np.max),
                          sr=sr, hop_length=hop_length,
                          x_axis='time', y_axis='linear', ax=axes[0])
axes[0].set_title('Standard Spectrogram\n(linear frequency, dB)')

img = librosa.display.specshow(librosa.power_to_db(mel_S, ref=np.max),
                                sr=sr, hop_length=hop_length,
                                x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title('Mel Spectrogram\n(mel frequency, dB)')
fig.colorbar(img, ax=axes[1], format='%+2.0f dB')

plt.suptitle('Linear vs Mel Frequency Spectrogram', fontsize=13)
plt.tight_layout(); plt.show()

print("\nMel gives more resolution at low frequencies")
print("and compresses high frequencies — matching perceptual sensitivity.")

## 5. Log-Mel Spectrogram — What Most Models Actually Use

**Why take the log?**

Loudness is perceived **logarithmically**.  
60 dB is not twice as loud as 30 dB — it is 1,000x more powerful in energy.

- Linear energy range: 0 to 10,000,000+ (huge, hard to train on)
- Log (dB) range: −80 to 0 dB (compact, normalized)

$$X_{\text{log-mel}} = 10 \cdot \log_{10}(\text{MelSpec} + \epsilon)$$

**Models that use Log-Mel:** Whisper, Tacotron 2, WaveNet, HuBERT, wav2vec 2.0

In [ ]:
# ── Log-Mel Spectrogram ──

log_mel = librosa.power_to_db(mel_S, ref=np.max)
# ref=np.max → loudest point = 0 dB → range is roughly [-80, 0] dB

print(f"Log-Mel shape:  {log_mel.shape}")
print(f"Value range:    [{log_mel.min():.1f}, {log_mel.max():.1f}] dB")

# Pipeline visualization: all 4 steps in one figure
fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(4, 1, hspace=0.45)
ax0, ax1, ax2, ax3 = [fig.add_subplot(gs[i]) for i in range(4)]

ax0.plot(np.linspace(0, duration, len(y)), y, color='steelblue', linewidth=0.5)
ax0.set_title('Step 1 — Waveform (raw samples)')
ax0.set_ylabel('Amplitude'); ax0.set_xlim(0, duration)

librosa.display.specshow(librosa.amplitude_to_db(np.abs(D), ref=np.max),
                          sr=sr, hop_length=hop_length, x_axis='time', y_axis='linear', ax=ax1)
ax1.set_title('Step 2 — STFT Power Spectrogram (linear frequency)')

img3 = librosa.display.specshow(mel_S, sr=sr, hop_length=hop_length,
                                  x_axis='time', y_axis='mel', ax=ax2)
ax2.set_title('Step 3 — Mel Spectrogram (mel frequency, linear power)')
fig.colorbar(img3, ax=ax2)

img4 = librosa.display.specshow(log_mel, sr=sr, hop_length=hop_length,
                                  x_axis='time', y_axis='mel', ax=ax3)
ax3.set_title('Step 4 — Log-Mel Spectrogram (mel frequency, dB scale)  <- model input')
fig.colorbar(img4, ax=ax3, format='%+2.0f dB')

plt.suptitle('Audio Feature Extraction Pipeline', fontsize=14, y=1.01)
plt.show()

print(f"\nFinal feature shape:  {log_mel.shape}  ({n_mels} mel bands x {log_mel.shape[1]} frames)")
print(f"Total values:          {n_mels * log_mel.shape[1]:,}  (vs {len(y):,} raw samples)")
print(f"Compression:           {len(y) / (n_mels * log_mel.shape[1]):.1f}x")

## 6. MFCC — Mel-Frequency Cepstral Coefficients

MFCCs are the classic feature for **traditional speech recognition** (before deep learning).  
Still used in GMM-HMM systems and speaker recognition pipelines.

**Pipeline:**
```
Log-Mel Spectrogram  ->  DCT (Discrete Cosine Transform)  ->  MFCC
```

DCT converts the mel bands into "cepstral coefficients" that describe the spectral envelope:
- Early coefficients = broad spectral shape (vowel identity, speaker characteristics)
- Later coefficients = fine detail (usually discarded)

**Why DCT?** To **decorrelate** features — adjacent mel bands are highly correlated.  
After DCT, coefficients are more independent, which suits Gaussian mixture models (GMM).

**Typically 13–40 coefficients are kept.**

| Feature | Size | Best for |
|---------|------|---------|
| Log-Mel Spectrogram | 80 x T | Whisper, Tacotron, WaveNet |
| MFCC | 13–40 x T | GMM-HMM ASR, speaker verification |
| Raw waveform | 16000 x T | wav2vec 2.0, HuBERT |

In [ ]:
# ── MFCC ──

n_mfcc = 40

mfcc = librosa.feature.mfcc(
    y=y, sr=sr, n_mfcc=n_mfcc,
    n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
)

print(f"MFCC shape: {mfcc.shape}  ({n_mfcc} coefficients x {mfcc.shape[1]} frames)")

mfcc_delta  = librosa.feature.delta(mfcc)           # velocity
mfcc_delta2 = librosa.feature.delta(mfcc, order=2)  # acceleration

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

for ax, data, title, cmap in zip(
    axes,
    [mfcc, mfcc_delta, mfcc_delta2],
    ['MFCC — static spectral shape',
     'Delta MFCC — velocity (rate of change over time)',
     'Delta-Delta MFCC — acceleration'],
    ['coolwarm', 'RdBu', 'PiYG']
):
    img = librosa.display.specshow(data, sr=sr, hop_length=hop_length,
                                    x_axis='time', ax=ax, cmap=cmap)
    ax.set_title(title)
    ax.set_ylabel('Coefficient')
    fig.colorbar(img, ax=ax)

plt.suptitle('MFCC + Delta Features — traditional ASR input', fontsize=13)
plt.tight_layout(); plt.show()

print("\nIn traditional ASR, concatenate [MFCC, delta, delta-delta]")
print(f"Each frame becomes a {n_mfcc * 3}-dimensional vector.")

In [ ]:
# ── Feature Comparison ──

print("=" * 62)
print("AUDIO FEATURE SUMMARY")
print("=" * 62)

features = {
    "Raw waveform":           (len(y),),
    "STFT magnitude":         D.shape,
    "Mel Spectrogram":        mel_S.shape,
    "Log-Mel Spectrogram":    log_mel.shape,
    "MFCC (40)":              mfcc.shape,
    "MFCC + delta + delta2":  (mfcc.shape[0] * 3, mfcc.shape[1]),
}

print(f"  {'Feature':<28} {'Shape':<18} {'Total':>12}")
print("  " + "-" * 60)
for name, shape in features.items():
    total = 1
    for s in shape: total *= s
    shape_str = " x ".join(str(s) for s in shape)
    print(f"  {name:<28} {shape_str:<18} {total:>12,}")

print()
print("  Which feature to use:")
print("    Whisper, Tacotron 2      ->  Log-Mel Spectrogram (80 mel bands)")
print("    Traditional GMM-HMM ASR  ->  MFCC + delta + delta-delta")
print("    wav2vec 2.0 / HuBERT     ->  Raw waveform (1D CNN frontend)")

## 7. Case Study: What Does Whisper Use as Input?

Whisper (OpenAI, 2022) is a state-of-the-art multilingual ASR model  
trained on 680,000 hours of audio from the internet.

**Feature extraction pipeline:**
```
Audio (any length)
    |  Resample to 16,000 Hz
    v
Pad or trim to exactly 30 seconds
    |  STFT  (n_fft=400, hop_length=160)
    v
Log-Mel Spectrogram  (80 mel bands)
    |
    v  Shape: (80, 3000)   <- 30s x 100 frames/s = 3,000 time frames
Transformer Encoder  (ViT-style patch embedding)
    |
    v
Text token output
```

**Why 80 mel bands?** Covers all speech-relevant content efficiently.  
**Why 100 frames/s?** hop_length = 160 -> 160 / 16,000 = 10 ms per frame.

![Mel Spectrogram Demo](./figures/mel_spectrogram_demo.png)

*Figure: Example Log-Mel Spectrogram (from torchaudio documentation)*

In [ ]:
# ── Replicating Whisper Preprocessing ──

def whisper_log_mel(audio, sr_original=16000):
    TARGET_SR = 16000
    N_FFT, HOP, N_MELS, CHUNK = 400, 160, 80, 30

    if sr_original != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr_original, target_sr=TARGET_SR)

    # Pad or trim to 30 seconds
    target_len = TARGET_SR * CHUNK
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio, sr=TARGET_SR,
        n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS,
        fmin=0.0, fmax=TARGET_SR // 2, power=2.0
    )

    # Log + normalize to roughly [-1, 1]
    log_mel = np.log10(np.maximum(mel, 1e-10))
    log_mel = np.maximum(log_mel, log_mel.max() - 8.0)
    log_mel = (log_mel + 4.0) / 4.0
    return log_mel.astype(np.float32)


feats = whisper_log_mel(y, sr_original=sr)
print(f"Output shape: {feats.shape}   (expected: (80, 3000))")
print(f"Value range:  [{feats.min():.3f}, {feats.max():.3f}]")

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

img = librosa.display.specshow(feats, x_axis='time', y_axis='mel',
                                sr=16000, hop_length=160, ax=axes[0],
                                fmax=8000, cmap='inferno')
axes[0].set_title('Whisper Log-Mel Spectrogram (80 mel bands x 3,000 frames)')
fig.colorbar(img, ax=axes[0])

img2 = librosa.display.specshow(feats[:, :300], x_axis='time', y_axis='mel',
                                  sr=16000, hop_length=160, ax=axes[1],
                                  fmax=8000, cmap='inferno')
axes[1].set_title('First 3 seconds — zoomed in')
fig.colorbar(img2, ax=axes[1])

plt.tight_layout(); plt.show()

## Summary

### Why Spectrograms Matter for Deep Learning

1. **Perceptually motivated** — Mel scale and dB scale model how humans hear
2. **Compact** — 16,000 raw samples/s -> ~100 frames/s (160x smaller)
3. **Time-frequency representation** — captures *what frequency, when* — ideal for Conv2D and attention
4. **Robust** — more noise-tolerant than raw waveform due to filterbank averaging

### Feature Cheat Sheet

| Feature | Output shape | Best for |
|---------|-------------|---------|
| **Waveform** | (T,) | wav2vec 2.0, HuBERT |
| **STFT** | (F, T) | intermediate step only |
| **Mel Spectrogram** | (M, T) | audio classification |
| **Log-Mel Spectrogram** | (M, T) | **Whisper, Tacotron 2, WaveNet** |
| **MFCC** | (C, T) | traditional ASR, speaker ID |

### Pipeline

```
Audio -> Resample -> STFT -> Mel Filterbank -> log -> Log-Mel -> Model
          16 kHz    n_fft    80 filters        dB     (80 x T)
```

### References

| | |
|--|--|
| Whisper | [arxiv 2212.04356](https://arxiv.org/abs/2212.04356) |
| torchaudio | [pytorch.org/audio](https://pytorch.org/audio/stable/) |
| librosa | [librosa.org](https://librosa.org/doc/latest/) |
| Mel scale | Stevens, Volkmann & Newman (1937) |